In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt

from pathlib import Path
import os
os.chdir("./..")

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import TimeSeriesSplit

pd.set_option("display.max_columns", None)

# Loading data

In [2]:
alarms = pd.read_csv("data/alarms/alarms_data_preprocessed_v2.csv")
weather = pd.read_csv("data/weather/weather_data_preprocessed_v3.csv")
isw = pd.read_csv("data/isw/isw_data_preprocessed_v5.csv")
telegram = pd.read_csv("data/telegram/telegram_hourly_features_v3.csv")

In [3]:
alarms.head(3)

,region_id,region_title,region_city,all_region,start,end,start_hour,start_minute
0,12,Львівська область,Львівська обл.,1,2022-02-24 07:43:17,2022-02-24 09:52:28,7,43
1,23,Чернігівська область,Чернігівська обл.,1,2022-02-24 14:00:43,2022-02-24 17:11:43,14,0
2,3,Вінницька область,Вінницька обл.,1,2022-02-24 15:40:42,2022-02-24 16:10:42,15,40


In [4]:
weather.head(3)

,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_preciptype,hour_windspeed,hour_winddir,hour_pressure,hour_visibility,hour_cloudcover,hour_uvindex,hour_conditions,real_hour_datetime,city
0,2.4,-1.6,89.18,0.8,0.0,0.0,['snow'],15.5,275.6,1020.0,0.0,91.5,0.0,Overcast,2022-02-24 00:00:00,Lutsk
1,2.4,-1.5,87.90,0.6,0.0,0.0,['snow'],14.8,280.3,1021.0,0.2,88.2,0.0,Partially cloudy,2022-02-24 01:00:00,Lutsk
2,2.9,-0.8,88.58,1.2,0.0,0.0,['snow'],14.4,310.0,1022.0,10.0,100.0,0.0,Overcast,2022-02-24 02:00:00,Lutsk


In [5]:
isw.head(3)

,text_length,isw_cluster_0,isw_cluster_1,isw_cluster_2,isw_cluster_3,isw_cluster_4,isw_cluster_5,isw_cluster_6,isw_cluster_7,isw_cluster_8,isw_cluster_9,anomaly_count_7d,avg_dist_centroid_7d,news_count_7d,topic_entropy_7d,topic_entropy_30d,date,news_velocity_30d,news_velocity_7d,centroid_shift_7d,avg_dist_centroid_30d,dom_cluster_share_7d,anomaly_count_30d,centroid_shift_30d,news_count_30d,dom_cluster_share_30d
0,13953,0,0,0,0,0,0,0,0,0,1,1,0.345617,7,0.598270,0.589003,2022-03-28,28,0,0.229950,0.396254,0.714286,3,0.565055,29,0.724138
1,10309,0,0,0,1,0,0,0,0,0,0,1,0.340311,7,0.410116,0.589003,2022-03-29,27,0,0.201195,0.393526,0.857143,3,0.527332,29,0.724138
2,30148,0,0,0,2,0,0,0,0,0,0,1,0.339411,7,0.598270,0.589003,2022-03-30,26,0,0.197885,0.387930,0.714286,3,0.512615,29,0.724138


In [6]:
telegram.head(3)

,datetime,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h,hour_of_day,day_of_week,is_weekend
0,2022-02-24 02:00:00+02:00,1.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1.0,1.0,0.0,2,3,0
1,2022-02-24 03:00:00+02:00,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1.0,1.0,0.0,3,3,0
2,2022-02-24 04:00:00+02:00,1.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2.0,2.0,0.0,4,3,0


# Preparing data for merging

All data have different granularity, so it need to be unified. I want to make granularity 1 hour, which would be best option for our task.

Key idea is to merge data by time and region id. Each row will have composed key created by `time` and `region_id` columns

## Alarms

In [7]:
alarms.head()

,region_id,region_title,region_city,all_region,start,end,start_hour,start_minute
0,12,Львівська область,Львівська обл.,1,2022-02-24 07:43:17,2022-02-24 09:52:28,7,43
1,23,Чернігівська область,Чернігівська обл.,1,2022-02-24 14:00:43,2022-02-24 17:11:43,14,0
2,3,Вінницька область,Вінницька обл.,1,2022-02-24 15:40:42,2022-02-24 16:10:42,15,40
3,19,Харківська область,Харківська обл.,1,2022-02-24 20:11:47,2022-02-24 20:59:47,20,11
4,18,Тернопільська область,Тернопільська обл.,1,2022-02-25 01:59:36,2022-02-25 09:00:19,1,59


In [8]:
alarms.info()

<class 'pandas.DataFrame'>
RangeIndex: 76147 entries, 0 to 76146
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   region_id     76147 non-null  int64
 1   region_title  76147 non-null  str  
 2   region_city   76147 non-null  str  
 3   all_region    76147 non-null  int64
 4   start         76147 non-null  str  
 5   end           76147 non-null  str  
 6   start_hour    76147 non-null  int64
 7   start_minute  76147 non-null  int64
dtypes: int64(4), str(4)
memory usage: 12.1 MB


Convert `start`, `end` columns to datetime format

In [9]:
alarms["start"] = pd.to_datetime(alarms["start"])
alarms["end"] = pd.to_datetime(alarms["end"])

In [10]:
alarms

,region_id,region_title,region_city,all_region,start,end,start_hour,start_minute
0,12,Львівська область,Львівська обл.,1,2022-02-24 07:43:17,2022-02-24 09:52:28,7,43
1,23,Чернігівська область,Чернігівська обл.,1,2022-02-24 14:00:43,2022-02-24 17:11:43,14,0
2,3,Вінницька область,Вінницька обл.,1,2022-02-24 15:40:42,2022-02-24 16:10:42,15,40
3,19,Харківська область,Харківська обл.,1,2022-02-24 20:11:47,2022-02-24 20:59:47,20,11
4,18,Тернопільська область,Тернопільська обл.,1,2022-02-25 01:59:36,2022-02-25 09:00:19,1,59
...,...,...,...,...,...,...,...,...
76142,9,Київська область,Київська обл.,1,2026-03-16 22:28:50,2026-03-16 23:18:03,22,28
76143,9,Київська область,Київ,0,2026-03-16 22:34:21,2026-03-16 23:17:13,22,34
76144,17,Сумська область,Сумська обл.,1,2026-03-16 22:37:12,2026-03-17 07:26:30,22,37
76145,19,Харківська область,Харківська обл.,1,2026-03-16 23:27:16,2026-03-17 00:18:41,23,27


In [11]:
alarms.isna().sum()

region_id       0
region_title    0
region_city     0
all_region      0
start           0
end             0
start_hour      0
start_minute    0
dtype: int64

In [12]:
alarms.start.min(), alarms.end.max()

(Timestamp('2022-02-24 07:43:17'), Timestamp('2026-03-17 07:26:30'))

Make 1 hour granularity

In [13]:
# create a helper column with the list of hours per alarm
alarms["hours"] = alarms.apply(
    lambda row: pd.date_range(row["start"].floor("h"), row["end"].floor("h"), freq="h"),
    axis=1
)

# explode - each hour becomes its own row, region_id stays attached
alarm_expanded = alarms[["region_id", "hours", "region_city"]].explode("hours")
alarm_expanded = alarm_expanded.rename(columns={"hours": "time"})
alarm_expanded["alarm"] = 1

alarm_expanded.head()

,region_id,time,region_city,alarm
0,12,2022-02-24 07:00:00,Львівська обл.,1
0,12,2022-02-24 08:00:00,Львівська обл.,1
0,12,2022-02-24 09:00:00,Львівська обл.,1
1,23,2022-02-24 14:00:00,Чернігівська обл.,1
1,23,2022-02-24 15:00:00,Чернігівська обл.,1


In [14]:
alarm_expanded.duplicated(subset=["region_id", "time"]).sum()

np.int64(13459)

In [15]:
alarm_expanded = alarm_expanded.drop_duplicates(subset=["region_id", "time"])

In [16]:
alarm_expanded.shape

(175534, 4)

Add column for count of neighbour regions alarms

In [17]:
%%time
# Neighbor adjacency map using region_ids from your CSV:
# 1=Crimea, 2=Vinnytsia, 3=Volyn, 4=Dnipropetrovsk, 5=Donetsk,
# 6=Zhytomyr, 7=Zakarpattia, 8=Zaporizhzhia, 9=Ivano-Frankivsk,
# 10=Kyiv, 11=Kirovohrad, 12=Luhansk, 13=Lviv, 14=Mykolaiv,
# 15=Odessa, 16=Poltava, 17=Rivne, 18=Sumy, 19=Ternopil,
# 20=Kharkiv, 21=Kherson, 22=Khmelnytskyi, 23=Cherkasy,
# 24=Chernivtsi, 25=Chernihiv

NEIGHBORS = {
    1:  [8, 14, 15],           # Crimea — Zaporizhzhia, Mykolaiv, Odessa (sea/land)
    2:  [6, 10, 23, 11, 15, 24, 22],  # Vinnytsia
    3:  [17, 13],              # Volyn
    4:  [16, 20, 8, 21, 14, 11, 5],   # Dnipropetrovsk
    5:  [20, 12, 8, 4],        # Donetsk
    6:  [10, 25, 17, 22, 2],   # Zhytomyr
    7:  [13, 9],               # Zakarpattia
    8:  [4, 5, 21],            # Zaporizhzhia
    9:  [13, 7, 24, 19],       # Ivano-Frankivsk
    10: [25, 18, 16, 23, 2, 6], # Kyiv
    11: [23, 16, 4, 14, 15, 2], # Kirovohrad
    12: [20, 5],               # Luhansk
    13: [3, 17, 19, 9, 7],     # Lviv
    14: [15, 21, 4, 11],       # Mykolaiv
    15: [14, 11, 2, 24],       # Odessa
    16: [18, 20, 4, 11, 23, 25, 10],  # Poltava
    17: [3, 13, 19, 22, 6],    # Rivne
    18: [25, 16, 20],          # Sumy
    19: [17, 22, 13, 9, 24],   # Ternopil
    20: [18, 16, 4, 5, 12],    # Kharkiv
    21: [14, 4, 8],            # Kherson
    22: [6, 2, 19, 17, 24],    # Khmelnytskyi
    23: [10, 16, 4, 11, 2, 25], # Cherkasy
    24: [9, 22, 2, 15, 19],    # Chernivtsi
    25: [10, 18, 16, 23],      # Chernihiv
}

alarm_lookup = alarm_expanded.set_index(["region_id", "time"])["alarm"]

def count_neighbor_alarms(row):
    neighbors = NEIGHBORS.get(row.region_id, [])
    return sum(alarm_lookup.get((n, row.time), 0) for n in neighbors)

alarm_expanded["neighbor_alarms_count"] = alarm_expanded.apply(count_neighbor_alarms, axis=1)
alarm_expanded.sample(5)

CPU times: total: 2min 4s
Wall time: 2min 11s


,region_id,time,region_city,alarm,neighbor_alarms_count
6627,14,2022-06-25 04:00:00,Одеська обл.,1,3
68202,5,2025-10-25 07:00:00,Донецька обл.,1,1
69671,8,2025-11-23 07:00:00,Запорізька обл.,1,2
7802,19,2022-07-25 00:00:00,Харківська обл.,1,0
66056,4,2025-09-11 13:00:00,Дніпропетровська обл.,1,2


## Weather

In [18]:
weather.head()

,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_preciptype,hour_windspeed,hour_winddir,hour_pressure,hour_visibility,hour_cloudcover,hour_uvindex,hour_conditions,real_hour_datetime,city
0,2.4,-1.6,89.18,0.8,0.0,0.0,['snow'],15.5,275.6,1020.0,0.0,91.5,0.0,Overcast,2022-02-24 00:00:00,Lutsk
1,2.4,-1.5,87.90,0.6,0.0,0.0,['snow'],14.8,280.3,1021.0,0.2,88.2,0.0,Partially cloudy,2022-02-24 01:00:00,Lutsk
2,2.9,-0.8,88.58,1.2,0.0,0.0,['snow'],14.4,310.0,1022.0,10.0,100.0,0.0,Overcast,2022-02-24 02:00:00,Lutsk
3,2.3,-1.3,86.63,0.3,0.0,0.0,['snow'],13.3,295.1,1021.0,0.1,92.0,0.0,Overcast,2022-02-24 03:00:00,Lutsk
4,1.9,-1.8,87.85,0.1,0.0,0.0,['snow'],13.3,305.8,1021.0,0.0,93.8,0.0,Overcast,2022-02-24 04:00:00,Lutsk


In [19]:
weather.real_hour_datetime.min(), weather.real_hour_datetime.max()

('2022-02-24 00:00:00', '2026-03-16 23:00:00')

In [20]:
weather.isna().sum()

hour_temp                  0
hour_feelslike             0
hour_humidity              0
hour_dew                   0
hour_precip                0
hour_precipprob            0
hour_preciptype       717975
hour_windspeed             0
hour_winddir               0
hour_pressure              0
hour_visibility            0
hour_cloudcover            0
hour_uvindex               0
hour_conditions            0
real_hour_datetime         0
city                       0
dtype: int64

In [21]:
weather.loc[weather.hour_preciptype.isna()].sample(3)

,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_preciptype,hour_windspeed,hour_winddir,hour_pressure,hour_visibility,hour_cloudcover,hour_uvindex,hour_conditions,real_hour_datetime,city
587692,0.7,-2.9,93.01,-0.3,0.0,0.0,NaN,11.5,128.8,1015.0,7.0,99.7,0.0,Overcast,2024-12-07 01:00:00,Lviv
1620,17.6,17.6,44.29,5.3,0.0,0.0,NaN,4.7,341.9,1017.0,24.1,65.8,8.0,Partially cloudy,2022-05-02 13:00:00,Lutsk
98787,26.0,26.0,28.15,6.2,0.0,0.0,NaN,7.2,7.0,1023.4,53.9,31.7,7.0,Partially cloudy,2022-07-18 16:00:00,Uzhgorod


In [22]:
weather["hour_preciptype"] = weather.hour_preciptype.fillna("None")

In [23]:
weather.isna().sum()

hour_temp             0
hour_feelslike        0
hour_humidity         0
hour_dew              0
hour_precip           0
hour_precipprob       0
hour_preciptype       0
hour_windspeed        0
hour_winddir          0
hour_pressure         0
hour_visibility       0
hour_cloudcover       0
hour_uvindex          0
hour_conditions       0
real_hour_datetime    0
city                  0
dtype: int64

### Adding `region_id` column

In [24]:
regions = pd.read_csv("data/alarms/regions.csv")

regions

,region,center_city_ua,center_city_en,region_alt,region_id
0,АР Крим,Сімферополь,Simferopol,Крим,1
1,Вінницька,Вінниця,Vinnytsia,Вінниччина,2
2,Волинська,Луцьк,Lutsk,Волинь,3
3,Дніпропетровська,Дніпро,Dnipro,Дніпропетровщина,4
4,Донецька,Донецьк,Donetsk,Донеччина,5
5,Житомирська,Житомир,Zhytomyr,Житомирщина,6
6,Закарпатська,Ужгород,Uzhgorod,Закарпаття,7
7,Запорізька,Запоріжжя,Zaporozhye,Запоріжжя,8
8,Івано-Франківська,Івано-Франківськ,Ivano-Frankivsk,Івано-Франківщина,9
9,Київська,Київ,Kyiv,Київщина,10


In [25]:
regions.center_city_en.unique()

<ArrowStringArray>
[     'Simferopol',       'Vinnytsia',           'Lutsk',          'Dnipro',
         'Donetsk',        'Zhytomyr',        'Uzhgorod',      'Zaporozhye',
 'Ivano-Frankivsk',            'Kyiv',   'Kropyvnytskyi',         'Luhansk',
            'Lviv',        'Mykolaiv',           'Odesa',         'Poltava',
           'Rivne',            'Sumy',        'Ternopil',         'Kharkiv',
         'Kherson',    'Khmelnytskyi',        'Cherkasy',      'Chernivtsi',
       'Chernihiv']
Length: 25, dtype: str

In [26]:
weather_df = weather.copy()

In [27]:
weather_df.city.unique()

<ArrowStringArray>
[          'Lutsk',   'Kropyvnytskyi',          'Dnipro',            'Kyiv',
         'Kherson',      'Chernivtsi',       'Chernihiv',           'Odesa',
        'Mykolaiv',         'Kharkiv',    'Khmelnytskyi',         'Donetsk',
        'Uzhgorod',      'Zaporozhye',           'Rivne',        'Zhytomyr',
        'Ternopil',         'Poltava',            'Lviv', 'Ivano-Frankivsk',
        'Cherkasy',            'Sumy',       'Vinnytsia']
Length: 23, dtype: str

In [28]:
regions.loc[~regions.center_city_en.isin(set(weather_df.city.unique()))]

,region,center_city_ua,center_city_en,region_alt,region_id
0,АР Крим,Сімферополь,Simferopol,Крим,1
11,Луганська,Луганськ,Luhansk,Луганщина,12


There is no Crimea and Luhansk in `weather` dataset. But knowing that in this cities permanent alarm status, it is not necessary to know weather there. Model should always predict 1 for alarm status there.

In [29]:
region_id = pd.DataFrame({"city": regions.center_city_en, "region_id": regions.region_id})
weather_df = weather_df.merge(region_id, on=["city"], how="left")

In [30]:
weather_df["time"] = pd.to_datetime(weather_df.pop("real_hour_datetime"))

In [31]:
weather_df

,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_preciptype,hour_windspeed,hour_winddir,hour_pressure,hour_visibility,hour_cloudcover,hour_uvindex,hour_conditions,city,region_id,time
0,2.4,-1.6,89.18,0.8,0.0,0.0,['snow'],15.5,275.6,1020.0,0.0,91.5,0.0,Overcast,Lutsk,3,2022-02-24 00:00:00
1,2.4,-1.5,87.90,0.6,0.0,0.0,['snow'],14.8,280.3,1021.0,0.2,88.2,0.0,Partially cloudy,Lutsk,3,2022-02-24 01:00:00
2,2.9,-0.8,88.58,1.2,0.0,0.0,['snow'],14.4,310.0,1022.0,10.0,100.0,0.0,Overcast,Lutsk,3,2022-02-24 02:00:00
3,2.3,-1.3,86.63,0.3,0.0,0.0,['snow'],13.3,295.1,1021.0,0.1,92.0,0.0,Overcast,Lutsk,3,2022-02-24 03:00:00
4,1.9,-1.8,87.85,0.1,0.0,0.0,['snow'],13.3,305.8,1021.0,0.0,93.8,0.0,Overcast,Lutsk,3,2022-02-24 04:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
808846,10.4,10.4,38.34,-3.2,0.0,0.0,None,8.6,148.6,1014.0,10.0,6.1,0.0,Clear,Lviv,13,2026-03-16 19:00:00
808847,9.3,8.1,42.51,-2.8,0.0,0.0,None,8.3,146.5,1016.0,10.0,46.7,0.0,Partially cloudy,Lviv,13,2026-03-16 20:00:00
808848,8.7,7.4,45.60,-2.4,0.0,0.0,None,8.3,157.4,1016.0,10.0,58.3,0.0,Partially cloudy,Lviv,13,2026-03-16 21:00:00
808849,8.2,6.7,48.23,-2.1,0.0,0.0,None,8.6,168.3,1016.0,10.0,58.1,0.0,Partially cloudy,Lviv,13,2026-03-16 22:00:00


## Telegram

In [32]:
telegram.head()

,datetime,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h,hour_of_day,day_of_week,is_weekend
0,2022-02-24 02:00:00+02:00,1.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1.0,1.0,0.0,2,3,0
1,2022-02-24 03:00:00+02:00,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1.0,1.0,0.0,3,3,0
2,2022-02-24 04:00:00+02:00,1.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2.0,2.0,0.0,4,3,0
3,2022-02-24 05:00:00+02:00,17.0,3.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,18.0,19.0,3.0,5,3,0
4,2022-02-24 06:00:00+02:00,7.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,25.0,26.0,-3.0,6,3,0


In [33]:
telegram.info()

<class 'pandas.DataFrame'>
RangeIndex: 35441 entries, 0 to 35440
Data columns (total 24 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   datetime                   35441 non-null  str    
 1   messages_count             35441 non-null  float64
 2   has_threat_sum             35441 non-null  float64
 3   nlp_артобстрілу            35441 non-null  int64  
 4   nlp_бпла                   35441 non-null  int64  
 5   nlp_відбій                 35441 non-null  int64  
 6   nlp_відбій_тривоги         35441 non-null  int64  
 7   nlp_дніпропетровська       35441 non-null  int64  
 8   nlp_донецька               35441 non-null  int64  
 9   nlp_запорізька             35441 non-null  int64  
 10  nlp_нікополь               35441 non-null  int64  
 11  nlp_нікополь_нікопольська  35441 non-null  int64  
 12  nlp_нікопольська           35441 non-null  int64  
 13  nlp_повітряна              35441 non-null  int64  
 14  n

In [34]:
tg_df = telegram.copy()
tg_df.rename({"datetime": "time"}, axis=1, inplace=True)
tg_df.time = pd.to_datetime(tg_df.time, utc=True) \
                .dt.tz_convert("Europe/Kyiv") \
                .dt.floor("h", ambiguous="infer")

In [35]:
tg_df.time.dtype

datetime64[us, Europe/Kyiv]

In [36]:
tg_df.duplicated(subset="time").sum()

np.int64(0)

## ISW

In [37]:
isw.head()

,text_length,isw_cluster_0,isw_cluster_1,isw_cluster_2,isw_cluster_3,isw_cluster_4,isw_cluster_5,isw_cluster_6,isw_cluster_7,isw_cluster_8,isw_cluster_9,anomaly_count_7d,avg_dist_centroid_7d,news_count_7d,topic_entropy_7d,topic_entropy_30d,date,news_velocity_30d,news_velocity_7d,centroid_shift_7d,avg_dist_centroid_30d,dom_cluster_share_7d,anomaly_count_30d,centroid_shift_30d,news_count_30d,dom_cluster_share_30d
0,13953,0,0,0,0,0,0,0,0,0,1,1,0.345617,7,0.598270,0.589003,2022-03-28,28,0,0.229950,0.396254,0.714286,3,0.565055,29,0.724138
1,10309,0,0,0,1,0,0,0,0,0,0,1,0.340311,7,0.410116,0.589003,2022-03-29,27,0,0.201195,0.393526,0.857143,3,0.527332,29,0.724138
2,30148,0,0,0,2,0,0,0,0,0,0,1,0.339411,7,0.598270,0.589003,2022-03-30,26,0,0.197885,0.387930,0.714286,3,0.512615,29,0.724138
3,45026,0,0,0,1,0,0,0,0,0,2,1,0.273573,7,0.598270,0.619376,2022-03-31,25,0,0.193133,0.377509,0.714286,3,0.526493,29,0.689655
4,14288,0,0,0,1,0,0,0,0,0,0,1,0.295166,7,0.598270,0.589003,2022-04-01,24,0,0.183463,0.370606,0.714286,3,0.547304,29,0.724138


In [38]:
isw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1227 entries, 0 to 1226
Data columns (total 26 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   text_length            1227 non-null   int64  
 1   isw_cluster_0          1227 non-null   int64  
 2   isw_cluster_1          1227 non-null   int64  
 3   isw_cluster_2          1227 non-null   int64  
 4   isw_cluster_3          1227 non-null   int64  
 5   isw_cluster_4          1227 non-null   int64  
 6   isw_cluster_5          1227 non-null   int64  
 7   isw_cluster_6          1227 non-null   int64  
 8   isw_cluster_7          1227 non-null   int64  
 9   isw_cluster_8          1227 non-null   int64  
 10  isw_cluster_9          1227 non-null   int64  
 11  anomaly_count_7d       1227 non-null   int64  
 12  avg_dist_centroid_7d   1227 non-null   float64
 13  news_count_7d          1227 non-null   int64  
 14  topic_entropy_7d       1227 non-null   float64
 15  topic_entropy_3

In [39]:
isw.date = pd.to_datetime(isw.date)

In [40]:
isw.date

0      2022-03-28
1      2022-03-29
2      2022-03-30
3      2022-03-31
4      2022-04-01
          ...    
1222   2026-03-06
1223   2026-03-07
1224   2026-03-08
1225   2026-03-09
1226   2026-03-10
Name: date, Length: 1227, dtype: datetime64[us]

In [41]:
isw.isna().sum()

text_length              0
isw_cluster_0            0
isw_cluster_1            0
isw_cluster_2            0
isw_cluster_3            0
isw_cluster_4            0
isw_cluster_5            0
isw_cluster_6            0
isw_cluster_7            0
isw_cluster_8            0
isw_cluster_9            0
anomaly_count_7d         0
avg_dist_centroid_7d     0
news_count_7d            0
topic_entropy_7d         0
topic_entropy_30d        0
date                     0
news_velocity_30d        0
news_velocity_7d         0
centroid_shift_7d        0
avg_dist_centroid_30d    0
dom_cluster_share_7d     0
anomaly_count_30d        0
centroid_shift_30d       0
news_count_30d           0
dom_cluster_share_30d    0
dtype: int64

In [42]:
isw.duplicated(subset="date").sum()

np.int64(0)

# Merging

## Creating spine

Idea is to create spine by multiplying hours by regions and then merge everythin to it by time and region_id

In [43]:
timeline = pd.date_range(
    alarms.start.min().floor("h"),
    alarms.end.max().ceil("h"),
    freq="h"
)

region_ids = alarms.region_id.unique()

expected_length = len(timeline) * len(region_ids)

print(f"Number of hours: {len(timeline)}")
print(f"Number of regions: {len(region_ids)}")
print()
print(f"Expected length of spine: {expected_length}")

Number of hours: 35570
Number of regions: 23

Expected length of spine: 818110


In [44]:
spine = pd.MultiIndex.from_product([region_ids, timeline], names=["region_id", "time"]) \
            .to_frame(index=False) \
            .sort_values(["region_id", "time"]) \
            .reset_index(drop=True)

In [45]:
spine.shape

(818110, 2)

In [46]:
spine.head()

,region_id,time
0,1,2022-02-24 07:00:00
1,1,2022-02-24 08:00:00
2,1,2022-02-24 09:00:00
3,1,2022-02-24 10:00:00
4,1,2022-02-24 11:00:00


## Merging alarms to spine

In [47]:
merged_df = spine.merge(alarm_expanded, on=["region_id", "time"], how="left")
merged_df["alarm"] = merged_df["alarm"].fillna(0).astype(int)

In [48]:
merged_df

,region_id,time,region_city,alarm,neighbor_alarms_count
0,1,2022-02-24 07:00:00,NaN,0,NaN
1,1,2022-02-24 08:00:00,NaN,0,NaN
2,1,2022-02-24 09:00:00,NaN,0,NaN
3,1,2022-02-24 10:00:00,NaN,0,NaN
4,1,2022-02-24 11:00:00,NaN,0,NaN
...,...,...,...,...,...
818105,24,2026-03-17 04:00:00,NaN,0,NaN
818106,24,2026-03-17 05:00:00,NaN,0,NaN
818107,24,2026-03-17 06:00:00,NaN,0,NaN
818108,24,2026-03-17 07:00:00,NaN,0,NaN


## Merging weather

In [49]:
merged_df = merged_df.merge(weather_df, on=["region_id", "time"], how="left")

In [50]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 818110 entries, 0 to 818109
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   region_id              818110 non-null  int64         
 1   time                   818110 non-null  datetime64[us]
 2   region_city            175534 non-null  str           
 3   alarm                  818110 non-null  int64         
 4   neighbor_alarms_count  175534 non-null  float64       
 5   hour_temp              737576 non-null  float64       
 6   hour_feelslike         737576 non-null  float64       
 7   hour_humidity          737576 non-null  float64       
 8   hour_dew               737576 non-null  float64       
 9   hour_precip            737576 non-null  float64       
 10  hour_precipprob        737576 non-null  float64       
 11  hour_preciptype        737576 non-null  str           
 12  hour_windspeed         737576 non-null  float64       


## Merging telegram

In [51]:
merged_df.time = merged_df.time.dt.tz_localize("Europe/Kyiv", ambiguous="NaT", nonexistent="NaT")

In [52]:
tg_df.shape

(35441, 24)

In [53]:
merged_df = merged_df.merge(tg_df, on=["time"], how="left")

In [54]:
merged_df.shape

(818110, 43)

## Merging ISW

In [55]:
isw.shape

(1227, 26)

In [56]:
merged_df["date"] = merged_df["time"].dt.date

In [57]:
merged_df.date = pd.to_datetime(merged_df.date)

In [58]:
merged_df = merged_df.merge(isw, on="date", how="left")

In [59]:
merged_df = merged_df.drop(columns="date")

In [60]:
merged_df.shape

(818110, 68)

## Result

In [61]:
merged_df.head()

,region_id,time,region_city,alarm,neighbor_alarms_count,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_preciptype,hour_windspeed,hour_winddir,hour_pressure,hour_visibility,hour_cloudcover,hour_uvindex,hour_conditions,city,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h,hour_of_day,day_of_week,is_weekend,text_length,isw_cluster_0,isw_cluster_1,isw_cluster_2,isw_cluster_3,isw_cluster_4,isw_cluster_5,isw_cluster_6,isw_cluster_7,isw_cluster_8,isw_cluster_9,anomaly_count_7d,avg_dist_centroid_7d,news_count_7d,topic_entropy_7d,topic_entropy_30d,news_velocity_30d,news_velocity_7d,centroid_shift_7d,avg_dist_centroid_30d,dom_cluster_share_7d,anomaly_count_30d,centroid_shift_30d,news_count_30d,dom_cluster_share_30d
0,1,2022-02-24 07:00:00+02:00,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,45.0,47.0,2.0,7.0,3.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2022-02-24 08:00:00+02:00,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,48.0,67.0,0.0,8.0,3.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,2022-02-24 09:00:00+02:00,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.0,86.0,-1.0,9.0,3.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,2022-02-24 10:00:00+02:00,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,52.0,99.0,0.0,10.0,3.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,2022-02-24 11:00:00+02:00,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,54.0,121.0,0.0,11.0,3.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [62]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 818110 entries, 0 to 818109
Data columns (total 68 columns):
 #   Column                     Non-Null Count   Dtype                      
---  ------                     --------------   -----                      
 0   region_id                  818110 non-null  int64                      
 1   time                       817926 non-null  datetime64[us, Europe/Kyiv]
 2   region_city                175534 non-null  str                        
 3   alarm                      818110 non-null  int64                      
 4   neighbor_alarms_count      175534 non-null  float64                    
 5   hour_temp                  737576 non-null  float64                    
 6   hour_feelslike             737576 non-null  float64                    
 7   hour_humidity              737576 non-null  float64                    
 8   hour_dew                   737576 non-null  float64                    
 9   hour_precip                737576 non-null  floa

In [63]:
merged_df.shape

(818110, 68)

In [64]:
print(f"Expected length: {expected_length}")
print(f"Actual length: {merged_df.shape[0]}")

Expected length: 818110
Actual length: 818110


# Preprocessing merged data

In [65]:
df = merged_df.copy()

In [66]:
df = df.loc[~df.time.isna()]

In [67]:
with pd.option_context("display.max_rows", None):
    print(df.isna().sum())

region_id                         0
time                              0
region_city                  642425
alarm                             0
neighbor_alarms_count        642425
hour_temp                     80433
hour_feelslike                80433
hour_humidity                 80433
hour_dew                      80433
hour_precip                   80433
hour_precipprob               80433
hour_preciptype               80433
hour_windspeed                80433
hour_winddir                  80433
hour_pressure                 80433
hour_visibility               80433
hour_cloudcover               80433
hour_uvindex                  80433
hour_conditions               80433
city                          80433
messages_count                 3082
has_threat_sum                 3082
nlp_артобстрілу                3082
nlp_бпла                       3082
nlp_відбій                     3082
nlp_відбій_тривоги             3082
nlp_дніпропетровська           3082
nlp_донецька                

In [68]:
df = df.drop(["city", "region_city"], axis=1)

In [70]:
df = df.loc[~df.hour_temp.isna()]

df.text_length = df.text_length.fillna(-1)
df.hour_preciptype = df.hour_preciptype.fillna("None")
df.neighbor_alarms_count = df.neighbor_alarms_count.fillna(0)

df.hour_visibility = df.hour_visibility.ffill()
df.hour_uvindex = df.hour_uvindex.ffill()

df = df.loc[~df.messages_count.isna()]

df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["day"] = df["time"].dt.day

In [71]:
with pd.option_context("display.max_rows", None):
    print(df.isna().sum())

region_id                         0
time                              0
alarm                             0
neighbor_alarms_count             0
hour_temp                         0
hour_feelslike                    0
hour_humidity                     0
hour_dew                          0
hour_precip                       0
hour_precipprob                   0
hour_preciptype                   0
hour_windspeed                    0
hour_winddir                      0
hour_pressure                     0
hour_visibility                   0
hour_cloudcover                   0
hour_uvindex                      0
hour_conditions                   0
messages_count                    0
has_threat_sum                    0
nlp_артобстрілу                   0
nlp_бпла                          0
nlp_відбій                        0
nlp_відбій_тривоги                0
nlp_дніпропетровська              0
nlp_донецька                      0
nlp_запорізька                    0
nlp_нікополь                

In [72]:
df = df.ffill() # using forward fill to fill missng values with values from previous records

In [73]:
with pd.option_context("display.max_rows", None):
    print(df.isna().sum())

region_id                      0
time                           0
alarm                          0
neighbor_alarms_count          0
hour_temp                      0
hour_feelslike                 0
hour_humidity                  0
hour_dew                       0
hour_precip                    0
hour_precipprob                0
hour_preciptype                0
hour_windspeed                 0
hour_winddir                   0
hour_pressure                  0
hour_visibility                0
hour_cloudcover                0
hour_uvindex                   0
hour_conditions                0
messages_count                 0
has_threat_sum                 0
nlp_артобстрілу                0
nlp_бпла                       0
nlp_відбій                     0
nlp_відбій_тривоги             0
nlp_дніпропетровська           0
nlp_донецька                   0
nlp_запорізька                 0
nlp_нікополь                   0
nlp_нікополь_нікопольська      0
nlp_нікопольська               0
nlp_повітр

In [74]:
df.loc[df.isw_cluster_1.isna(), "time"].unique()

<DatetimeArray>
['2022-02-24 07:00:00+02:00', '2022-02-24 08:00:00+02:00',
 '2022-02-24 09:00:00+02:00', '2022-02-24 10:00:00+02:00',
 '2022-02-24 11:00:00+02:00', '2022-02-24 12:00:00+02:00',
 '2022-02-24 13:00:00+02:00', '2022-02-24 14:00:00+02:00',
 '2022-02-24 15:00:00+02:00', '2022-02-24 16:00:00+02:00',
 ...
 '2022-03-27 14:00:00+03:00', '2022-03-27 15:00:00+03:00',
 '2022-03-27 16:00:00+03:00', '2022-03-27 17:00:00+03:00',
 '2022-03-27 18:00:00+03:00', '2022-03-27 19:00:00+03:00',
 '2022-03-27 20:00:00+03:00', '2022-03-27 21:00:00+03:00',
 '2022-03-27 22:00:00+03:00', '2022-03-27 23:00:00+03:00']
Length: 760, dtype: datetime64[us, Europe/Kyiv]

missing values only at first month of war, so we can drop them.

In [75]:
df = df.dropna(axis=0)

In [76]:
df.isna().sum().sum()

np.int64(0)

In [77]:
df = df.reset_index(drop=True)

In [78]:
encoder = LabelEncoder()

cat_cols = list(df.select_dtypes(include=["str"]).columns)

for col_name in cat_cols:
    df[col_name] = encoder.fit_transform(df[col_name])

In [79]:
df.head(3)

,region_id,time,alarm,neighbor_alarms_count,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_preciptype,hour_windspeed,hour_winddir,hour_pressure,hour_visibility,hour_cloudcover,hour_uvindex,hour_conditions,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h,hour_of_day,day_of_week,is_weekend,text_length,isw_cluster_0,isw_cluster_1,isw_cluster_2,isw_cluster_3,isw_cluster_4,isw_cluster_5,isw_cluster_6,isw_cluster_7,isw_cluster_8,isw_cluster_9,anomaly_count_7d,avg_dist_centroid_7d,news_count_7d,topic_entropy_7d,topic_entropy_30d,news_velocity_30d,news_velocity_7d,centroid_shift_7d,avg_dist_centroid_30d,dom_cluster_share_7d,anomaly_count_30d,centroid_shift_30d,news_count_30d,dom_cluster_share_30d,year,month,day
0,2,2022-03-28 00:00:00+03:00,0,0.0,0.4,-2.2,43.82,-10.5,0.0,0.0,0,7.6,314.0,1029.0,24.1,2.0,0.0,0,13.0,5.0,0.0,0.0,4.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,5.0,5.0,4.0,0.0,86.0,289.0,-3.0,0.0,0.0,0.0,13953.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.345617,7.0,0.59827,0.589003,28.0,0.0,0.22995,0.396254,0.714286,3.0,0.565055,29.0,0.724138,2022,3,28
1,2,2022-03-28 01:00:00+03:00,0,0.0,-0.1,-2.5,45.43,-10.5,0.0,0.0,0,6.8,308.6,1029.0,24.1,4.2,0.0,0,8.0,2.0,0.0,0.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,4.0,0.0,45.0,285.0,-3.0,1.0,0.0,0.0,13953.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.345617,7.0,0.59827,0.589003,28.0,0.0,0.22995,0.396254,0.714286,3.0,0.565055,29.0,0.724138,2022,3,28
2,2,2022-03-28 02:00:00+03:00,0,0.0,-0.6,-2.3,47.87,-10.3,0.0,0.0,0,5.0,305.7,1029.0,24.1,3.1,0.0,0,7.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,28.0,269.0,-2.0,2.0,0.0,0.0,13953.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.345617,7.0,0.59827,0.589003,28.0,0.0,0.22995,0.396254,0.714286,3.0,0.565055,29.0,0.724138,2022,3,28


In [80]:
df.sort_values(by=["region_id", "time"])

,region_id,time,alarm,neighbor_alarms_count,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_preciptype,hour_windspeed,hour_winddir,hour_pressure,hour_visibility,hour_cloudcover,hour_uvindex,hour_conditions,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h,hour_of_day,day_of_week,is_weekend,text_length,isw_cluster_0,isw_cluster_1,isw_cluster_2,isw_cluster_3,isw_cluster_4,isw_cluster_5,isw_cluster_6,isw_cluster_7,isw_cluster_8,isw_cluster_9,anomaly_count_7d,avg_dist_centroid_7d,news_count_7d,topic_entropy_7d,topic_entropy_30d,news_velocity_30d,news_velocity_7d,centroid_shift_7d,avg_dist_centroid_30d,dom_cluster_share_7d,anomaly_count_30d,centroid_shift_30d,news_count_30d,dom_cluster_share_30d,year,month,day
0,2,2022-03-28 00:00:00+03:00,0,0.0,0.4,-2.2,43.82,-10.5,0.0,0.0,0,7.6,314.0,1029.0,24.1,2.0,0.0,0,13.0,5.0,0.0,0.0,4.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,5.0,5.0,4.0,0.0,86.0,289.0,-3.0,0.0,0.0,0.0,13953.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.345617,7.0,0.598270,0.589003,28.0,0.0,0.22995,0.396254,0.714286,3.0,0.565055,29.0,0.724138,2022,3,28
1,2,2022-03-28 01:00:00+03:00,0,0.0,-0.1,-2.5,45.43,-10.5,0.0,0.0,0,6.8,308.6,1029.0,24.1,4.2,0.0,0,8.0,2.0,0.0,0.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,4.0,0.0,45.0,285.0,-3.0,1.0,0.0,0.0,13953.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.345617,7.0,0.598270,0.589003,28.0,0.0,0.22995,0.396254,0.714286,3.0,0.565055,29.0,0.724138,2022,3,28
2,2,2022-03-28 02:00:00+03:00,0,0.0,-0.6,-2.3,47.87,-10.3,0.0,0.0,0,5.0,305.7,1029.0,24.1,3.1,0.0,0,7.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,28.0,269.0,-2.0,2.0,0.0,0.0,13953.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.345617,7.0,0.598270,0.589003,28.0,0.0,0.22995,0.396254,0.714286,3.0,0.565055,29.0,0.724138,2022,3,28
3,2,2022-03-28 03:00:00+03:00,0,0.0,-1.1,-1.1,51.24,-9.9,0.0,0.0,0,4.0,289.8,1029.0,24.1,1.8,0.0,0,11.0,10.0,0.0,0.0,0.0,0.0,2.0,0.0,2.0,0.0,0.0,0.0,9.0,9.0,9.0,0.0,2.0,26.0,275.0,10.0,3.0,0.0,0.0,13953.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.345617,7.0,0.598270,0.589003,28.0,0.0,0.22995,0.396254,0.714286,3.0,0.565055,29.0,0.724138,2022,3,28
4,2,2022-03-28 04:00:00+03:00,0,0.0,-1.4,-1.4,52.80,-9.8,0.0,0.0,0,3.6,245.2,1029.0,24.1,0.0,0.0,0,9.0,1.0,0.0,0.0,6.0,6.0,2.0,0.0,2.0,0.0,0.0,0.0,1.0,1.0,2.0,6.0,2.0,27.0,283.0,-9.0,4.0,0.0,0.0,13953.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.345617,7.0,0.598270,0.589003,28.0,0.0,0.22995,0.396254,0.714286,3.0,0.565055,29.0,0.724138,2022,3,28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734228,24,2026-03-11 14:00:00+02:00,0,0.0,15.0,15.0,24.78,-5.0,0.0,0.0,0,18.4,120.0,1023.0,10.0,0.0,0.0,0,19.0,6.0,6.0,0.0,8.0,4.0,0.0,0.0,0.0,15.0,15.0,15.0,2.0,2.0,6.0,4.0,0.0,66.0,457.0,3.0,14.0,2.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.225396,7.0,0.410116,0.388734,0.0,0.0,0.31613,0.332111,0.857143,3.0,0.281373,30.0,0.900000,2026,3,11
734229,24,2026-03-11 15:00:00+02:00,0,0.0,16.0,16.0,21.54,-6.0,0.0,0.0,0,16.6,120.0,1023.0,10.0,0.0,0.0,0,13.0,0.0,0.0,0.0,3.0,3.0,0.0,0.0,0.0,2.0,2.0,2.0,7.0,7.0,7.0,3.0,2.0,50.0,456.0,-6.0,15.0,2.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.225396,7.0,0.410116,0.388734,0.0,0.0,0.31613,0.332111,0.857143,3.0,0.281373,30.0,0.900000,2026,3,11
734230,24,2026-03-11 16:00:00+02:00,0,0.0,15.0,15.0,24.78,-5.0,0.0,0.0,0,25.9,120.0,1022.0,10.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,32.0,439.0,0.0,16.0,2.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0

In [81]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 734233 entries, 0 to 734232
Data columns (total 69 columns):
 #   Column                     Non-Null Count   Dtype                      
---  ------                     --------------   -----                      
 0   region_id                  734233 non-null  int64                      
 1   time                       734233 non-null  datetime64[us, Europe/Kyiv]
 2   alarm                      734233 non-null  int64                      
 3   neighbor_alarms_count      734233 non-null  float64                    
 4   hour_temp                  734233 non-null  float64                    
 5   hour_feelslike             734233 non-null  float64                    
 6   hour_humidity              734233 non-null  float64                    
 7   hour_dew                   734233 non-null  float64                    
 8   hour_precip                734233 non-null  float64                    
 9   hour_precipprob            734233 non-null  floa

In [84]:
cols_to_int = ['messages_count', 'has_threat_sum',
       'nlp_артобстрілу', 'nlp_бпла', 'nlp_відбій', 'nlp_відбій_тривоги',
       'nlp_дніпропетровська', 'nlp_донецька', 'nlp_запорізька',
       'nlp_нікополь', 'nlp_нікополь_нікопольська', 'nlp_нікопольська',
       'nlp_повітряна', 'nlp_повітряна_тривога', 'nlp_тривога', 'nlp_тривоги',
       'nlp_харківська', 'msg_count_last_3h', 'msg_count_last_24h',
       'threat_diff_1h', 'hour_of_day', 'day_of_week', 'is_weekend',
       'text_length', 'isw_cluster_0', 'isw_cluster_1', 'isw_cluster_2',
       'isw_cluster_3', 'isw_cluster_4', 'isw_cluster_5', 'isw_cluster_6',
       'isw_cluster_7', 'isw_cluster_8', 'isw_cluster_9', "news_count_7d", 
        "news_count_30d", "anomaly_count_7d", "anomaly_count_30d",
        "news_velocity_7d", "news_velocity_30d", "neighbor_alarms_count"]

df[cols_to_int] = df[cols_to_int].astype(int)

In [85]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 734233 entries, 0 to 734232
Data columns (total 69 columns):
 #   Column                     Non-Null Count   Dtype                      
---  ------                     --------------   -----                      
 0   region_id                  734233 non-null  int64                      
 1   time                       734233 non-null  datetime64[us, Europe/Kyiv]
 2   alarm                      734233 non-null  int64                      
 3   neighbor_alarms_count      734233 non-null  int64                      
 4   hour_temp                  734233 non-null  float64                    
 5   hour_feelslike             734233 non-null  float64                    
 6   hour_humidity              734233 non-null  float64                    
 7   hour_dew                   734233 non-null  float64                    
 8   hour_precip                734233 non-null  float64                    
 9   hour_precipprob            734233 non-null  floa

# Saving result

In [86]:
save_path = Path("data/merged/merged_v5.csv")

df.to_csv(save_path, index=False)